Инициализация глобальных переменных, достаем датасет

In [1]:
import os
import time
import json
import torch
import warnings
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
for warn in [UserWarning, FutureWarning]: warnings.filterwarnings("ignore", category = warn)

from src.data_utils.config import DatasetConfig
from src.data_utils.dataset_params import DatasetName
from src.data_utils.dataset_generator import DatasetGenerator
from src.models.models import TransformerClassifier

MAX_SEQ_LEN = 300
EMBEDDING_DIM = 64 # уменьшили: 128 -> 64, чтобы влезло в гит
BATCH_SIZE = 64 # подняли batch_size: 32 -> 64
LEARNING_RATE = 7e-5
NUM_EPOCHS = 100 # подняли количество эпох: 20 -> 100
NUM_CLASSES = 2

SAVE_DIR = "../pretrained"
os.makedirs(SAVE_DIR, exist_ok=True)
MODEL_SAVE_PATH = os.path.join(SAVE_DIR, "best_model.pth")
VOCAB_SAVE_PATH = os.path.join(SAVE_DIR, "vocab.json")
CONFIG_SAVE_PATH = os.path.join(SAVE_DIR, "config.json")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_TO_TRAIN = 'Transformer' 

config = DatasetConfig(
    load_from_disk=True,
    path_to_data="../datasets",
    train_size=25000,  # взяли весь датасет
    val_size=12500,
    test_size=12500
)
generator = DatasetGenerator(DatasetName.IMDB, config=config)
(X_train, y_train), (X_val, y_val), (X_test, y_test) = generator.generate_dataset()
VOCAB_SIZE = len(generator.vocab)

Создаем даталоадеры

In [2]:
def create_dataloader(X, y, batch_size):
    dataset = TensorDataset(torch.tensor(X, dtype=torch.long), torch.tensor(y, dtype=torch.long))
    return DataLoader(dataset, batch_size=batch_size, shuffle=True)
train_loader = create_dataloader(X_train, y_train, BATCH_SIZE)
val_loader = create_dataloader(X_val, y_val, BATCH_SIZE)

Инициализация модели

In [3]:
model_params = {'vocab_size': len(generator.vocab), 'embed_dim': EMBEDDING_DIM, 'num_heads': 8, 'num_layers': 4, 'num_classes': 2, 'max_seq_len': MAX_SEQ_LEN}
model = TransformerClassifier(**model_params)

Обучение

In [4]:
model.to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

best_val_loss = float('inf')
print(f"--- Начало обучения модели ---")
for epoch in range(NUM_EPOCHS):
    model.train()
    start_time = time.time()
    total_train_loss = 0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()
    avg_train_loss = total_train_loss / len(train_loader)
    
    model.eval()
    total_val_loss, correct_val, total_val = 0, 0, 0
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            total_val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_val += batch_y.size(0)
            correct_val += (predicted == batch_y).sum().item()
    avg_val_loss = total_val_loss / len(val_loader)
    val_accuracy = correct_val / total_val

    epoch_time = time.time() - start_time
    print(f"Эпоха {epoch+1}/{NUM_EPOCHS} | Время: {epoch_time:.2f}с | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_accuracy:.4f}")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"  -> Модель сохранена, новая лучшая Val Loss: {best_val_loss:.4f}")

--- Начало обучения модели ---
Эпоха 1/100 | Время: 14.62с | Train Loss: 0.6563 | Val Loss: 0.6460 | Val Acc: 0.6544
  -> Модель сохранена, новая лучшая Val Loss: 0.6460
Эпоха 2/100 | Время: 14.10с | Train Loss: 0.5749 | Val Loss: 0.5673 | Val Acc: 0.7217
  -> Модель сохранена, новая лучшая Val Loss: 0.5673
Эпоха 3/100 | Время: 14.13с | Train Loss: 0.5058 | Val Loss: 0.5285 | Val Acc: 0.7533
  -> Модель сохранена, новая лучшая Val Loss: 0.5285
Эпоха 4/100 | Время: 14.09с | Train Loss: 0.4664 | Val Loss: 0.4980 | Val Acc: 0.7724
  -> Модель сохранена, новая лучшая Val Loss: 0.4980
Эпоха 5/100 | Время: 14.22с | Train Loss: 0.4382 | Val Loss: 0.4785 | Val Acc: 0.7851
  -> Модель сохранена, новая лучшая Val Loss: 0.4785
Эпоха 6/100 | Время: 14.29с | Train Loss: 0.4166 | Val Loss: 0.4775 | Val Acc: 0.7814
  -> Модель сохранена, новая лучшая Val Loss: 0.4775
Эпоха 7/100 | Время: 14.14с | Train Loss: 0.3974 | Val Loss: 0.4636 | Val Acc: 0.7893
  -> Модель сохранена, новая лучшая Val Loss: 0.4

Снимем качество на тестовых данных из исходного датасета

In [5]:
def evaluate_on_test(model, test_loader, device, criterion):
    model.eval()
    total_test_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            total_test_loss += loss.item()
            
            _, predicted = torch.max(outputs.data, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(batch_y.cpu().numpy())
            
    avg_test_loss = total_test_loss / len(test_loader)
        
    accuracy = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary')
    
    return {'loss': avg_test_loss, 'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1_score': f1}


test_loader = create_dataloader(X_test, y_test, BATCH_SIZE)
test_metrics = evaluate_on_test(model, test_loader, DEVICE, criterion)
print(f"Метрики на тестовой выборке (из обучаемого датасета) итоговой модели\n{test_metrics}")

Метрики на тестовой выборке (из обучаемого датасета) итоговой модели
{'loss': 0.8565363820110049, 'accuracy': 0.8276, 'precision': 0.851743686651778, 'recall': 0.79328, 'f1_score': 0.8214729517024273}


Снимем качество на тестовых данных нового датасета. Считаем данные

In [6]:
text_processor = generator.get_text_processor()
config_polarity = DatasetConfig(
    load_from_disk=True,
    path_to_data="../datasets",
    train_size=25000,  # взяли весь датасет
    val_size=12500,
    test_size=12500,
    build_vocab=False
)
generator_polarity = DatasetGenerator(DatasetName.POLARITY, config=config_polarity)
generator_polarity.vocab = generator.vocab
generator_polarity.id2word = generator.id2word
generator_polarity.text_processor = text_processor
(X_train, y_train), (X_val, y_val), (X_test, y_test) = generator_polarity.generate_dataset()


In [7]:
test_loader = create_dataloader(X_test, y_test, BATCH_SIZE)

Посмтрим на метрики

In [8]:
test_metrics = evaluate_on_test(model, test_loader, DEVICE, criterion)
print(f"Метрики на тестовой выборке (из неизвестного датасета) итоговой модели\n{test_metrics}")

Метрики на тестовой выборке (из неизвестного датасета) итоговой модели
{'loss': 0.6786724476485836, 'accuracy': 0.73816, 'precision': 0.7227414330218068, 'recall': 0.7762906309751434, 'f1_score': 0.7485595759391565}


В целом видно, что модель что-то, да выучила. Гипотезы по улучшению:
 - Больше и разнообразнее данные для обучения
    - Чем больше словарь - тем лучше
 - Нужно чтобы тестовый датасет был больше похож на обучаемый

Сохранение итоговой модели

In [9]:
with open(VOCAB_SAVE_PATH, 'w', encoding='utf-8') as f:
    json.dump(generator.vocab, f, ensure_ascii=False, indent=4)

config = {
    "model_type": MODEL_TO_TRAIN,
    "max_seq_len": MAX_SEQ_LEN,
    "model_params": model_params,
}
with open(CONFIG_SAVE_PATH, 'w', encoding='utf-8') as f:
    json.dump(config, f, ensure_ascii=False, indent=4)
print(f"Конфигурация модели сохранена в: {CONFIG_SAVE_PATH}")

Конфигурация модели сохранена в: ../pretrained/config.json
